# 用真实 DeepSeek API 学习 fox_ai：一次模型调用实验室

**模型：`deepseek-flash` · 地址：`https://api.deepseek.com` · 主路径全部经过本地 `fox_ai`。**

这份 Notebook 是 [架构学习文档](ARCHITECTURE_STUDY.md) 的动手配套。每节先说明“观察什么”，再运行一小段实验。模型回答、思考增量、token 用量和工具请求均来自真实 API；工具是你在本地实际执行的 Python 计算函数。

| 实验 | 你需要学会什么 |
| --- | --- |
| 1. 环境与模型 | 导入本地包；区分 provider、api、模型名称 |
| 2. 请求参数 | 区分“支持思考”和“本次是否思考” |
| 3. 真实流式调用 | 观察 delta、内容块和最终 AssistantMessage |
| 4. 开启思考 | 区分服务端思考内容与最终答案 |
| 5. 思考强度 | 对同一问题比较 low、high，按需运行 max |
| 6. 用量与延迟 | input/output/cache/reasoning 的关系与首字延迟 |
| 7. 完整结果与多轮 | complete() 如何复用链路；历史由谁维护 |
| 8. 工具闭环 | Tool → ToolCall → 本地执行 → ToolResultMessage → 回答 |
| 9. 截断与失败 | length 不等于完整回答，错误消息如何判断 |
| 10. 复盘 | 把实验结果对应到架构和源码 |

按顺序运行。默认生成 **6 次真实模型请求**，重跑调用单元格会再次请求；`max` 和截断实验默认关闭，打开后各增加一次。每次调用有输出上限，重试关闭。保留的输出是一次真实执行样本，不保证重跑后逐字相同。

使用 VS Code 的 Jupyter 功能或 JupyterLab，选择项目的 **Python 3.14** 内核。Notebook 直接使用顶层 `await`，不要改成 `asyncio.run()`。


## 1. 环境、源码导入与密钥

项目公共入口会导入 OpenAI 和 Anthropic 两个适配器，因此都需要安装，即使这次只调用 DeepSeek。首次使用先在所选内核安装依赖：取消下一格第一行 `%pip` 的注释并单独运行，完成后恢复注释；若旧模块已加载，重启内核。

依赖版本写在同目录 [requirements-learning.txt](requirements-learning.txt)，用于复现本次验证环境。Notebook 不自动安装依赖，也不更改你已有的环境配置。


In [1]:
import importlib.util
import sys
from importlib.metadata import version

if sys.version_info < (3, 14):
    raise RuntimeError("请选择 Python 3.14 内核，以与项目 pyproject.toml 一致。")
required = ["openai", "anthropic", "pydantic", "httpx", "IPython"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f"当前内核缺少 {missing}；先运行上面的 %pip 安装行。")
print("Python:", sys.version.split()[0])
print({name: version(name) for name in ["openai", "anthropic", "pydantic", "httpx"]})


Python: 3.14.7
{'openai': '3.19.0', 'anthropic': '1.8.0', 'pydantic': '2.13.5', 'httpx': '0.28.1'}


In [2]:
import asyncio
import getpass
import html
import json
import os
import re
from collections import Counter
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
from time import perf_counter

import httpx
from IPython.display import HTML, display
from pydantic import BaseModel, ConfigDict, Field

# 支持从仓库根目录、packages/fox_ai 或其子目录启动。
REPO_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "packages/fox_ai/src/stream.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("请在 FoxCode 仓库内启动 Notebook。")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# 当前是源码目录，不使用尚未配置好的 from pi_ai import ...。
from packages.fox_ai import src as fox_ai
from packages.fox_ai.src import (
    AssistantMessage, Context, Model, StreamOptions, TextContent,
    ThinkingContent, Tool, ToolCall, ToolResultMessage, UserMessage,
)

print("已导入本地 fox_ai；可用协议实现：",
      type(fox_ai.get_api_provider("openai-completions")).__name__)


已导入本地 fox_ai；可用协议实现： _OpenAIProvider


密钥只从环境变量 `DEEPSEEK_API_KEY` 或下面的隐藏输入读取，不写入 Notebook。这里不会展示完整 Options 或请求头，因为它们可能包含凭据。重启内核后，若未设置环境变量，需要重新输入一次。


In [3]:
_deepseek_api_key = os.environ.get("DEEPSEEK_API_KEY", "").strip()
if not _deepseek_api_key:
    _deepseek_api_key = getpass.getpass("输入 DeepSeek API key（隐藏输入）：").strip()
if not _deepseek_api_key:
    raise RuntimeError("API key 不能为空。")

BASE_URL = "https://api.deepseek.com"
MODEL_ID = "deepseek-flash"
print("密钥已载入；目标：", BASE_URL, MODEL_ID)


密钥已载入；目标： https://api.deepseek.com deepseek-flash


### 注册模型：能力、协议、调用配置不要混在一起

- `provider="deepseek"` 表示服务归属；`api="openai-completions"` 表示使用哪套适配器。
- `reasoning=True` 是模型能力描述，**不是本次请求的思考开关**。
- `max_tokens=4096` 是本实验选用的默认输出预算，不是服务端最大能力。
- 本实验不配置费率，所以 `usage.cost` 的默认零值不能解读成免费；这里只统计真实 token 用量。

我们使用普通接口上的普通工具，不开启 beta strict 模式。

### 你提供的 pi-agent 配置，哪些值得沿用？

| 配置 | 在当前 Python fox_ai 中如何理解 |
| --- | --- |
| provider 的 baseUrl / api | 对应 Model.base_url / api；密钥单独进入 Options |
| models 中的 id、name、input、reasoning | 可用于构造 Model，但此次仍使用你最初指定的 deepseek-flash |
| contextWindow / maxTokens | 模型元数据；每次实验仍单独限制生成 token，不直接请求几十万 token |
| cost | 可以承载费率，旧配置不作为当前账单依据，本实验暂不配置 |
| requiresReasoningContentOnAssistantMessages | 当前适配器没有读取此键；设为 false 也不能改变服务端历史要求 |
| thinkingFormat / thinkingLevelMap | coding-agent 直接把 thinkingLevelMap 载入 Model.thinking_level_map；fox_ai 根据 thinkingFormat 生成 provider 请求 |

你贴出的 effort 映射把 minimal、low、medium、high 都变成 high，因此无法用它比较不同思考强度。本实验通过明确的请求参数分别传 low / high / max。运行时的通用级别会先经过模型映射，再由 provider 转成实际请求字段。


In [4]:
model = Model(
    id=MODEL_ID,
    name="DeepSeek Flash · fox_ai 学习实验",
    provider="deepseek",
    api="openai-completions",
    base_url=BASE_URL,
    input=["text"],
    reasoning=True,
    max_tokens=4096,
    compat={"supportsStrictMode": False},
)
fox_ai.register_model(model)
assert fox_ai.get_model("deepseek", MODEL_ID) is model
print({"id": model.id, "provider": model.provider,
       "api": model.api, "reasoning_capability": model.reasoning})


{'id': 'deepseek-flash', 'provider': 'deepseek', 'api': 'openai-completions', 'reasoning_capability': True}


## 2. 构造请求选项：怎样真正打开或关闭思考？

在本次核对的 [DeepSeek 思考模式文档](https://api-docs.deepseek.com/guides/thinking_mode/) 中，开关是 `thinking.type`，强度是 `reasoning_effort`，使用 `low / high / max`。通过 OpenAI SDK 时，额外的 `thinking` 字段应放在 `extra_body`。

本地 OpenAI 适配器会合并 `sampling_params` 后调用 SDK，所以使用下面这个明确的映射。**只设置 `SimpleStreamOptions(reasoning="high")` 尚不能让当前代码自动产生 DeepSeek 的同名厂商参数。**

| 你的意图 | 传给 SDK 的配置 |
| --- | --- |
| 关闭思考 | `extra_body={"thinking": {"type": "disabled"}}` |
| 开启思考 | `extra_body={"thinking": {"type": "enabled"}}` |
| 调整强度 | `reasoning_effort="low" / "high" / "max"` |

`max_tokens` 控制生成预算；effort 表达思考强度偏好，二者不是同一个参数。为了让实验易观察，不自动重试，也不设置与思考模式无关的温度实验。


In [5]:
def make_options(*, thinking=False, effort="low", max_tokens=2048, tool_choice=None):
    if effort not in {"low", "high", "max"}:
        raise ValueError("本实验的 effort 使用 low / high / max。")
    sdk_params = {
        "extra_body": {"thinking": {"type": "enabled" if thinking else "disabled"}}
    }
    if thinking:
        sdk_params["reasoning_effort"] = effort
    if tool_choice is not None:
        sdk_params["tool_choice"] = tool_choice
    return StreamOptions(
        api_key=_deepseek_api_key,
        max_tokens=max_tokens,
        timeout_ms=60_000,
        max_retries=0,
        sampling_params=sdk_params,
    )

# 只显示不含密钥的参数片段。
print("关闭：", make_options(thinking=False).sampling_params)
print("开启：", make_options(thinking=True, effort="high").sampling_params)


关闭： {'extra_body': {'thinking': {'type': 'disabled'}}}
开启： {'extra_body': {'thinking': {'type': 'enabled'}}, 'reasoning_effort': 'high'}


### 一个小型观察器：把同一套测量用于所有实验

先定义展示与统计函数。这些代码不调用模型。记录的是普通标量，避免把事件的可变 `partial` 当成历史快照。

后面的 `observe_stream()` 内部最重要的三步仍然是：`fox_ai.stream()` → `async for event` → `await event_stream.result()`。它额外记录首个内容时间、首个答案时间和用量；HTTP 客户端由 Notebook 用 `async with` 管理。


In [6]:
RUNS = []


def show_table(rows):
    if not rows:
        return
    columns = list(rows[0])
    esc = lambda value: html.escape(str(value))
    head = "".join(f"<th>{esc(c)}</th>" for c in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{esc(row.get(c, ''))}</td>" for c in columns) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))


def text_of(message):
    return "".join(b.text for b in message.content if isinstance(b, TextContent))


def thinking_of(message):
    return "".join(b.thinking for b in message.content if isinstance(b, ThinkingContent))


def check_result(message, *, allow_length=False):
    if message.stop_reason in {"error", "aborted", "pending"}:
        detail = (message.error_message or message.stop_reason).replace(_deepseek_api_key, "[hidden]")
        detail = re.sub(r"sk-[A-Za-z0-9_-]{12,}", "[hidden]", detail)
        raise RuntimeError(f"模型调用未成功：{detail[:800]}")
    if message.stop_reason == "length" and not allow_length:
        raise RuntimeError("生成达到上限，内容可能不完整；请增大 max_tokens 后重跑本格。")


def record_run(label, message, elapsed, first_content=None, first_answer=None):
    usage = message.usage
    row = {
        "实验": label, "终态": message.stop_reason,
        "总秒数": round(elapsed, 3),
        "首内容秒数": None if first_content is None else round(first_content, 3),
        "首答案秒数": None if first_answer is None else round(first_answer, 3),
        "输入未缓存": usage.input, "缓存读取": usage.cache_read,
        "缓存写入": usage.cache_write, "输出": usage.output,
        "思考token(若报告)": usage.reasoning,
        "总token": usage.total_tokens,
        "思考字符": len(thinking_of(message)), "答案字符": len(text_of(message)),
    }
    RUNS.append(row)
    return row


In [7]:
async def observe_stream(context, options, label, *, show_thinking=False, allow_length=False):
    started = perf_counter()
    first_content = first_answer = None
    event_counts = Counter()
    trace = []
    channel = None
    thinking_printed = 0
    print(f"[{label}]")

    async with httpx.AsyncClient(timeout=60.0) as http_client:
        request_options = options.model_copy(update={"http_client": http_client})
        # 外层限时是避免无限等待；它不能替代 fox_ai 完整的生产任务取消机制。
        async with asyncio.timeout(180):
            event_stream = fox_ai.stream(model, context, request_options)
            async for event in event_stream:
                elapsed = perf_counter() - started
                event_counts[event.type] += 1
                delta = getattr(event, "delta", "")
                trace.append({
                    "事件": event.type, "内容块": getattr(event, "content_index", None),
                    "相对秒数": round(elapsed, 3), "增量字符数": len(delta),
                })
                if delta and event.type in {"text_delta", "thinking_delta", "toolcall_delta"}:
                    if first_content is None:
                        first_content = elapsed
                if event.type == "text_delta":
                    if first_answer is None:
                        first_answer = elapsed
                    if channel != "text":
                        print("\n【最终答案】")
                        channel = "text"
                    print(delta, end="", flush=True)
                elif event.type == "thinking_delta" and show_thinking:
                    if channel != "thinking":
                        print("\n【服务端返回的思考内容，最多展示前 1200 字符】")
                        channel = "thinking"
                    visible = delta[:max(0, 1200 - thinking_printed)]
                    print(visible, end="", flush=True)
                    thinking_printed += len(visible)
                elif event.type == "toolcall_end":
                    call = event.tool_call
                    print("\n【工具请求】", call.id, call.name,
                          json.dumps(call.arguments, ensure_ascii=False))
            message = await event_stream.result()

    elapsed = perf_counter() - started
    print("\n终态：", message.stop_reason, "；事件：", dict(event_counts))
    record_run(label, message, elapsed, first_content, first_answer)
    check_result(message, allow_length=allow_length)
    return message, trace


## 3. 第一次真实流式调用：先关闭思考

**观察目标：**文字是否逐步出现，什么时候收到 `text_start / text_delta / text_end / done`，最后的消息和增量有什么关系。

下面一格会发起 **1 次真实请求**。Provider 已经在后台拼接完整消息，UI 只负责消费事件并显示增量。


In [8]:
intro_user = UserMessage(content="用三句话向初学者解释：什么是大模型的流式输出？")
intro_context = Context(
    system_prompt="你是一位简明的中文编程老师。回答准确，不超过150字。",
    messages=[intro_user],
)
intro_message, intro_trace = await observe_stream(
    intro_context, make_options(thinking=False, max_tokens=512), "流式·关闭思考"
)
assert intro_message.stop_reason == "stop"
assert text_of(intro_message)
assert not thinking_of(intro_message), "服务端返回了思考内容，请核对开关。"


[流式·关闭思考]

【最终答案】
1. 流式输出就是模型一边生成文字，一边立刻把已生成的部分发给你，而不是全部写完再一次性发送。  
2. 就像有人边想边说，你能实时看到文字逐个出现，体验更流畅。  
3. 技术上，它通过分段返回结果来实现，适合聊天、写作等需要快速响应的场景。
终态： stop ；事件： {'start': 1, 'text_start': 1, 'text_delta': 73, 'text_end': 1, 'done': 1}


In [9]:
show_table(intro_trace[:8] + intro_trace[-3:])
print("最终内容块类型：", [block.type for block in intro_message.content])
print("原始停止原因：", intro_message.raw_stop_reason)
print("问题：如果只保存 text_delta，会不会丢掉工具调用、用量和终态？")


事件,内容块,相对秒数,增量字符数
start,None,0.349,0
text_start,0,1.126,0
text_delta,0,1.126,1
text_delta,0,1.214,1
text_delta,0,1.225,1
text_delta,0,1.226,1
text_delta,0,1.241,1
text_delta,0,1.242,2
text_delta,0,1.624,1
text_end,0,1.625,0


最终内容块类型： ['text']
原始停止原因： stop
问题：如果只保存 text_delta，会不会丢掉工具调用、用量和终态？


## 4. 开启思考：区分思考内容与最终答案

**观察目标：**开启后是否出现 `thinking_*` 事件；先有内容和先有最终答案是否是同一时间。

只展示最多 1200 个思考字符以保持可读；完整返回仍保存在 `reasoning_low_message` 的内容块里。这里展示的是 DeepSeek 服务端输出的字段。

下面的题目也用于下一节对比强度。生成具有随机性，一道题的一次调用不能证明哪个档位整体更好。


In [10]:
REASONING_PROMPT = (
    "一个盒子有红球和蓝球共21个，红球是蓝球的2倍。"
    "取走3个红球，再放入3个蓝球，现在两种球各有多少？"
    "最终答案只给数量及一句核对。"
)
reasoning_context = Context(messages=[UserMessage(content=REASONING_PROMPT)])
reasoning_low_message, reasoning_low_trace = await observe_stream(
    reasoning_context,
    make_options(thinking=True, effort="low", max_tokens=2048),
    "思考·low",
    show_thinking=True,
)
assert text_of(reasoning_low_message)
assert thinking_of(reasoning_low_message), "本次未观察到思考内容，请检查服务端响应。"


[思考·low]

【服务端返回的思考内容，最多展示前 1200 字符】
我们需要回答中文。需要解题。注意题目：盒子红蓝共21，红球是蓝球2倍。设蓝x，红2x，总3x=21 => x=7，红14，蓝7。取走3红，放入3蓝：红11，蓝10。现在两种球各多少？红球11，蓝球10。最终答案只给数量及一句核对。需要只给数量及一句核对。可能要简洁。确保“最终答案只给数量及一句核对” 意思答案包括数量及一句核对，不要过程。可以：红球11个，蓝球10个。核对：取走后红14-3=11，蓝7+3=10。 这算一句核对。 但“只给数量及一句核对”也许先数量，再一句核对。 final. 
【最终答案】
红球11个，蓝球10个。核对：14−3=11，7+3=10。
终态： stop ；事件： {'start': 1, 'thinking_start': 1, 'thinking_delta': 177, 'text_start': 1, 'text_delta': 24, 'text_end': 1, 'thinking_end': 1, 'done': 1}


## 5. 思考强度：保持问题一致，再修改 effort

把上节的 `low` 改为 `high`，其它主要输入保持一致。重点比较请求是否生效、思考块、输出用量和延迟，不要求数字单调增加。网络与缓存也会影响耗时。

`max` 示例另设开关，默认不请求。输出预算过低时，思考可能占满预算而没有答案；这时应看到 `length`，不能把“没有最终文本”说成模型没有返回内容。


In [11]:
reasoning_high_message, reasoning_high_trace = await observe_stream(
    reasoning_context,
    make_options(thinking=True, effort="high", max_tokens=2048),
    "思考·high",
    show_thinking=True,
)
assert text_of(reasoning_high_message)
assert thinking_of(reasoning_high_message)
show_table([row for row in RUNS if row["实验"] in {"思考·low", "思考·high"}])


[思考·high]

【服务端返回的思考内容，最多展示前 1200 字符】
我们需要回答中文。需要解题。必须注意用户要求“最终答案只给数量及一句核对”。需要只给数量及一句核对。题目：一个盒子有红球和蓝球共21个，红球是蓝球的2倍。取走3个红球，再放入3个蓝球，现在两种球各有多少？

先算初始：红+蓝=21，红=2蓝。3蓝=21，蓝=7，红=14。取走3红：红11。放入3蓝：蓝10。总数仍21。现在红11，蓝10。核对一句：11+10=21，且红比蓝多1? 初始红是蓝2倍已满足14=2*7。需要最终答案只给数量及一句核对。可以是：“红球11个，蓝球10个。核对：11+10=21，且14-3=11、7+3=10。” 但“一句核对”可能要求一句。最终答案只给数量及一句核对，可以：“红球11个，蓝球10个。核对：11+10=21，符合操作后总数仍为21。” 这没有核对初始条件？可以包含。但只给数量及一句核对。应该简洁。

Need ensure no extra. final in Chinese. 
【最终答案】
红球11个，蓝球10个。核对：11+10=21，且14-3=11、7+3=10。
终态： stop ；事件： {'start': 1, 'thinking_start': 1, 'thinking_delta': 277, 'text_start': 1, 'text_delta': 31, 'text_end': 1, 'thinking_end': 1, 'done': 1}


实验,终态,总秒数,首内容秒数,首答案秒数,输入未缓存,缓存读取,缓存写入,输出,思考token(若报告),总token,思考字符,答案字符
思考·low,stop,1.351,0.481,1.281,79,0,0,202,177,281,249,30
思考·high,stop,1.925,0.726,1.842,79,0,0,309,277,388,415,40


In [12]:
RUN_MAX_EFFORT = False  # 想继续比较时，改为 True，单独重跑这一格。
if RUN_MAX_EFFORT:
    reasoning_max_message, _ = await observe_stream(
        reasoning_context,
        make_options(thinking=True, effort="max", max_tokens=4096),
        "思考·max", show_thinking=True,
    )
else:
    print("跳过 max；需要时把 RUN_MAX_EFFORT 改为 True。")


跳过 max；需要时把 RUN_MAX_EFFORT 改为 True。


## 6. 用量与延迟：数字应该怎样读？

下面只是查看已有结果，**不发请求**。

| 字段 | fox_ai 的含义 |
| --- | --- |
| input | 未命中缓存的输入 token |
| cache_read / cache_write | 缓存读 / 写 token；来自服务端统计的归一化 |
| output | 输出 token；reasoning 若存在是其中的子集 |
| reasoning | 服务端单独报告的思考 token，可能为 None |
| total_tokens | input + output + cache_read + cache_write |
| 首内容 | 首个非空文本、思考或工具参数增量到达的时间 |
| 首答案 | 首个 text_delta 到达的时间；纯工具调用可能没有 |

当前适配器把 `reasoning_tokens=0` 也归成 None，因此这里的 None 只说明“没有可用的独立正数统计”，不能据此推断没有思考，更不能把字符数当 token 数。

首答案时间包含连接与服务端生成等待；本实验没有单独测网络耗时。费用需要另行配置准确费率，查看 [官方模型与价格](https://api-docs.deepseek.com/quick_start/pricing/)。


In [13]:
show_table(RUNS)
usage = reasoning_high_message.usage
assert usage.total_tokens == usage.input + usage.output + usage.cache_read + usage.cache_write
if usage.reasoning is not None:
    assert usage.reasoning <= usage.output
print("高强度调用的用量：", usage.model_dump(exclude={"cost"}))
print("费用：本实验未配置费率，不把默认的 0 美元当作真实费用。")


实验,终态,总秒数,首内容秒数,首答案秒数,输入未缓存,缓存读取,缓存写入,输出,思考token(若报告),总token,思考字符,答案字符
流式·关闭思考,stop,1.625,1.126,1.126,34,0,0,73,None,107,0,122
思考·low,stop,1.351,0.481,1.281,79,0,0,202,177,281,249,30
思考·high,stop,1.925,0.726,1.842,79,0,0,309,277,388,415,40


高强度调用的用量： {'input': 79, 'output': 309, 'cache_read': 0, 'cache_write': 0, 'cache_write_1h': None, 'reasoning': 277, 'total_tokens': 388}
费用：本实验未配置费率，不把默认的 0 美元当作真实费用。


## 7. 补充：complete() 与多轮上下文

这次不展示增量，而是 `await complete()` 等待最终消息。底层仍然复用流式路径。

同时追问第一轮流式回答：历史必须由调用方明确放进 Context。模型不会因为你复用了同一个 Model 对象，就自动记住上次的问题。

下面会发起 **1 次真实请求**。这个实验不带工具，因此不需要回传此前的思考内容。


In [14]:
followup_context = Context(
    system_prompt=intro_context.system_prompt,
    messages=[intro_user, intro_message,
              UserMessage(content="沿用上面的解释，举一个聊天界面的例子，只写一句话。")],
)
started = perf_counter()
async with httpx.AsyncClient(timeout=60.0) as http_client:
    options = make_options(thinking=False, max_tokens=512).model_copy(
        update={"http_client": http_client}
    )
    async with asyncio.timeout(180):
        followup_message = await fox_ai.complete(model, followup_context, options)
record_run("complete·多轮", followup_message, perf_counter() - started)
check_result(followup_message)
assert text_of(followup_message)
print(text_of(followup_message))
print("提交的角色顺序：", [m.role for m in followup_context.messages])


就像你在聊天框输入问题后，AI的回复不是等几秒突然整段蹦出来，而是一个字一个字往外蹦，你眼睛跟着读就行。
提交的角色顺序： ['user', 'assistant', 'user']


## 8. 工具闭环：模型请求工具，不等于模型执行工具

我们定义一个真实的本地订单计算工具，使用 Decimal 计算折后总价，没有额外天气服务或虚构数据。

流程：**提供 Tool 描述 → 模型生成 ToolCall → Python 校验并执行 → 构造 ToolResultMessage → 再次请求模型**。这是两次模型调用，中间夹着一次本地工具执行。

当前 fox_ai 会读取 DeepSeek 的 `reasoning_content`，但历史转换没有把它写回。带工具的思考模式需要回传该字段，所以本节在两次请求中都**显式关闭思考**，并使用独立上下文。这个限制来自本地适配实现，不是 DeepSeek 不支持思考工具调用。参见 [思考与工具历史要求](https://api-docs.deepseek.com/guides/thinking_mode/)。


In [15]:
order_tool = Tool(
    name="calculate_order_total",
    description="按单价、数量和折扣计算人民币订单总价，按四舍五入保留两位小数。",
    parameters={
        "type": "object",
        "properties": {
            "unit_price": {"type": "number", "description": "人民币单价，如129.90"},
            "quantity": {"type": "integer", "description": "购买数量，正整数"},
            "discount": {"type": "number", "description": "折扣系数，85折为0.85"},
        },
        "required": ["unit_price", "quantity", "discount"],
        "additionalProperties": False,
    },
)


class OrderArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")
    unit_price: Decimal = Field(ge=0, le=1_000_000)
    quantity: int = Field(ge=1, le=10_000, strict=True)
    discount: Decimal = Field(ge=0, le=1)


def calculate_order_total(arguments):
    args = OrderArguments.model_validate(arguments)
    subtotal = args.unit_price * args.quantity
    total = (subtotal * args.discount).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
    return {"subtotal": str(subtotal), "discount": str(args.discount),
            "total": str(total), "currency": "CNY"}

# 仅注册允许执行的函数；不执行模型生成的任意 Python。
TOOL_HANDLERS = {"calculate_order_total": calculate_order_total}
print("已定义工具：", order_tool.name)


已定义工具： calculate_order_total


### 8.1 让模型生成工具调用

为了稳定观察工具流程，这一次通过 SDK 支持的具名 `tool_choice` 指定工具。普通应用通常使用自动选择。本地 `sampling_params` 会将该值合并进请求，具体字段可对照 [Chat Completions 接口](https://api-docs.deepseek.com/api/create-chat-completion/)。

注意输出中的调用 ID、工具名、参数与 `toolUse` 终态。参数增量只用于预览，本例等整个请求成功结束后再执行。


In [16]:
tool_user = UserMessage(content=(
    "请调用 calculate_order_total：商品单价129.90元，购买3件，85折。"
    "以工具结果为准，告诉我最终应付金额。"
))
tool_context = Context(messages=[tool_user], tools=[order_tool])
tool_request_message, tool_trace = await observe_stream(
    tool_context,
    make_options(
        thinking=False, max_tokens=1024,
        tool_choice={"type": "function", "function": {"name": order_tool.name}},
    ),
    "工具·请求参数",
)
calls = [block for block in tool_request_message.content if isinstance(block, ToolCall)]
assert tool_request_message.stop_reason == "toolUse" and calls
assert all(call.id for call in calls)
assert len({call.id for call in calls}) == len(calls)
show_table([{"id": call.id, "name": call.name, "arguments": call.arguments} for call in calls])


[工具·请求参数]

【工具请求】 call_00_90phLZpSAe6PCwhAryy92765 calculate_order_total {"unit_price": 129.9, "quantity": 3, "discount": 0.85}

终态： toolUse ；事件： {'start': 1, 'toolcall_start': 1, 'toolcall_delta': 26, 'toolcall_end': 1, 'done': 1}


id,name,arguments
call_00_90phLZpSAe6PCwhAryy92765,calculate_order_total,"{'unit_price': 129.9, 'quantity': 3, 'discount': 0.85}"


### 8.2 本地执行，然后构造 ToolResultMessage

这一格**不发模型请求**。你会看到工具的实际返回值。工具异常使用 `is_error=True` 表达，调用 ID 必须与模型给出的 ID 相同。

即使模型生成了结构化参数，也要进行执行前校验。Notebook 只开放上面一个计算函数。


In [17]:
tool_results = []
for call in calls:
    try:
        if call.name not in TOOL_HANDLERS:
            raise ValueError(f"没有允许执行的工具：{call.name}")
        payload = TOOL_HANDLERS[call.name](call.arguments)
        is_error = False
    except (ValueError, TypeError) as exc:
        payload = {"error": str(exc)}
        is_error = True
    tool_results.append(ToolResultMessage(
        tool_call_id=call.id,
        tool_name=call.name,
        content=[TextContent(text=json.dumps(payload, ensure_ascii=False))],
        is_error=is_error,
    ))
    print("工具实际返回：", payload, "；is_error：", is_error)

assert all(result.tool_call_id == call.id for call, result in zip(calls, tool_results))


工具实际返回： {'subtotal': '389.7', 'discount': '0.85', 'total': '331.25', 'currency': 'CNY'} ；is_error： False


### 8.3 把调用与结果一起回传，得到最终回答

必须保留 `assistant(toolCall)` 和对应的 `toolResult`，不能只把结果当成一条新的用户消息。

这一步再次关闭思考，并设置 `tool_choice="none"`，让教学流程在两次请求内结束。真正的 Agent 可以选择继续允许调用工具，但应设置轮次上限。


In [18]:
tool_followup_context = Context(
    messages=[tool_user, tool_request_message, *tool_results],
    tools=[order_tool],
)
tool_final_message, tool_final_trace = await observe_stream(
    tool_followup_context,
    make_options(thinking=False, max_tokens=512, tool_choice="none"),
    "工具·结果回传",
)
assert tool_final_message.stop_reason == "stop"
assert not any(isinstance(b, ToolCall) for b in tool_final_message.content)
assert text_of(tool_final_message)
print("上下文角色顺序：", [m.role for m in tool_followup_context.messages])
tool_rows = [row for row in RUNS if row["实验"].startswith("工具·")]
show_table(tool_rows)
print("本次内核中工具阶段累计 token：", sum(row["总token"] for row in tool_rows))


[工具·结果回传]

【最终答案】
按工具计算结果，最终应付金额为 **331.25 元**（CNY）。

- 商品单价：129.90 元
- 购买数量：3 件
- 原小计：389.70 元
- 折扣：85 折
- 折后应付：331.25 元
终态： stop ；事件： {'start': 1, 'text_start': 1, 'text_delta': 70, 'text_end': 1, 'done': 1}
上下文角色顺序： ['user', 'assistant', 'toolResult']


实验,终态,总秒数,首内容秒数,首答案秒数,输入未缓存,缓存读取,缓存写入,输出,思考token(若报告),总token,思考字符,答案字符
工具·请求参数,toolUse,1.026,0.953,None,254,128,0,71,None,453,0,0
工具·结果回传,stop,0.962,0.572,0.572,155,0,0,70,None,225,0,103


本次内核中工具阶段累计 token： 678


## 9. 补充：截断、错误和重试应该如何理解？

`stop` 表示正常结束，`toolUse` 表示转入工具处理，`length` 表示输出到达上限，`error` 表示失败。`await result()` 返回对象，不保证生成成功，必须检查停止原因。

可选的下一格用小预算观察截断。如果服务端在预算内完成，也可能返回 `stop`，不要把示例写成永远要求 `length` 的断言。

本实验不故意发送错误凭据或重复制造限流。实际遇到 401 应检查密钥；遇到 400 先检查参数与历史格式；短暂的 429/5xx 可考虑有限重试。输出已经出现后重试会生成新的尝试，不是原流的断点续传，工具副作用不能跟着盲目重做。


In [19]:
RUN_TRUNCATION = False
if RUN_TRUNCATION:
    truncated_message, _ = await observe_stream(
        Context(messages=[UserMessage(content="请详细讲解HTTP请求全过程，至少1000字。")]),
        make_options(thinking=False, max_tokens=16),
        "可选·小预算截断", allow_length=True,
    )
    print("停止原因：", truncated_message.stop_reason)
else:
    print("跳过截断实验；需要时把 RUN_TRUNCATION 改为 True。")


跳过截断实验；需要时把 RUN_TRUNCATION 改为 True。


## 10. 复盘：把实验对应到架构

| 实验中做的事 | 对应源码与设计 |
| --- | --- |
| 创建 Model，按 API 分发 | [models.py](src/models.py)：配置与协议实现分开 |
| 组装 Context，追问与回传结果 | [types.py](src/types.py)：显式维护历史 |
| stream / complete | [stream.py](src/stream.py)：共同调用链、两种消费方式 |
| text/thinking/toolcall 事件 | [events.py](src/events.py)：统一生成过程 |
| 等事件与等最终结果 | [event_stream.py](src/event_stream.py)：Queue + Future |
| 将 DeepSeek 参数发给兼容接口 | [openai_provider.py](src/providers/openai_provider.py)：协议转换 |
| 参数校验和函数执行 | 本 Notebook：上层执行器的责任，不属于 Provider |

自己尝试回答：

1. `model.reasoning=True` 和 `thinking.type="enabled"` 为什么不是一回事？
2. 为什么有了思考内容，却可能暂时没有最终答案？
3. 为什么不能直接把思考字符数当思考 token？
4. 工具闭环为什么至少包含两次模型请求？
5. 为什么工具结果要携带 `tool_call_id`，而不只是工具名？
6. 如果把两次工具请求都切成思考模式，当前适配器缺失哪一步？
7. `complete()` 真的是另一个非流式厂商接口吗？

可以优先扩展：更多轮工具循环、执行失败反馈、持久化 Context。先理解当前链路，再处理思考历史回传与统一取消。


In [20]:
show_table(RUNS)
print("本次内核记录的调用次数：", len(RUNS))
print("累计报告 token：", sum(row["总token"] for row in RUNS))
print("只汇总本次内核已记录结果；不是账户账单，也不含未报告用量的失败尝试。")


实验,终态,总秒数,首内容秒数,首答案秒数,输入未缓存,缓存读取,缓存写入,输出,思考token(若报告),总token,思考字符,答案字符
流式·关闭思考,stop,1.625,1.126,1.126,34,0,0,73,None,107,0,122
思考·low,stop,1.351,0.481,1.281,79,0,0,202,177,281,249,30
思考·high,stop,1.925,0.726,1.842,79,0,0,309,277,388,415,40
complete·多轮,stop,1.023,None,None,126,0,0,34,None,160,0,52
工具·请求参数,toolUse,1.026,0.953,None,254,128,0,71,None,453,0,0
工具·结果回传,stop,0.962,0.572,0.572,155,0,0,70,None,225,0,103


本次内核记录的调用次数： 6
累计报告 token： 1614
只汇总本次内核已记录结果；不是账户账单，也不含未报告用量的失败尝试。


### 运行问题与实验边界

- **缺少依赖**：在当前 Notebook 内核执行第 1 节安装行；项目根配置尚未列全本实验所需依赖。
- **找不到源码**：从 FoxCode 仓库内启动，按顺序执行导入单元格。
- **`length` 且没有答案**：增大 `max_tokens`，或先降低 effort；本格重复执行会再次请求。
- **工具阶段 400**：保持本节独立上下文和 `thinking=False`，不要夹入前面思考实验的历史。
- **超时或手动中断**：当前 fox_ai 生产任务取消尚不完整，必要时重启内核再运行；本实验不是取消机制示范。
- **重复运行**：重跑模型单元格会继续增加 RUNS；重新运行观察器定义单元格可清空统计，已有服务器计费不会被清除。

本 Notebook 不打印密钥；分享前也应检查自己后来新增的单元格是否输出了 Options、环境变量或请求头。

参数核对日期：2026-09-23。文档入口：[思考模式](https://api-docs.deepseek.com/guides/thinking_mode/)、[工具调用](https://api-docs.deepseek.com/guides/tool_calls/)、[Chat Completions](https://api-docs.deepseek.com/api/create-chat-completion/)、[模型与用量价格](https://api-docs.deepseek.com/quick_start/pricing/)。
